# LeetCode 916 — Word Subsets bằng Python

Link đề: https://leetcode.com/problems/word-subsets/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, String

Skeleton LeetCode:

```python
class Solution:
    def wordSubsets(self, words1: List[str], words2: List[str]) -> List[str]:
        ...
```

Ghi chú: kết quả là **danh sách các từ**, đề cho phép **thứ tự bất kỳ** nên test so sánh sau khi `sorted`. Điểm cốt lõi: "`b` là tập con của `a` kể cả số lần lặp" nghĩa là với mỗi chữ cái `c`, `a` có **ít nhất** `count_b(c)` chữ `c`. Một từ `a` thoả **mọi** `b` trong `words2` ⇔ với mỗi chữ cái, `count_a(c) ≥ max_b count_b(c)`. Vậy **gộp `words2` thành một vector yêu cầu `need[26]` = max theo từng chữ cái**, rồi kiểm tra từng từ của `words1` một lần. Oracle: **định nghĩa trực tiếp** (với từng cặp `(a, b)` tiêu thụ chữ cái bằng cách xoá khỏi danh sách).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: `b` là tập con (đa tập) của `a`; từ `a ∈ words1` là **universal** nếu **mọi** `b ∈ words2` là tập con của `a`.
2. Nắm công cụ Python: `collections.Counter`, phép `|` và `-` của `Counter` (max theo khoá, trừ), mảng đếm 26 phần tử, `str.count`, `all`.
3. Đi từ **kiểm tra từng cặp (`O(|w1|·|w2|)` lần kiểm)** → **gộp `words2` thành vector max** → **dùng phép toán `Counter`** → **dùng `str.count` chỉ cho các chữ cái cần**.
4. Hiểu mấu chốt: điều kiện "với mọi `b`: `count_a(c) ≥ count_b(c)`, với mọi `c`" tương đương "với mọi `c`: `count_a(c) ≥ max_b count_b(c)`" (hoán đổi hai lượng từ "với mọi" cho cùng một `a`).
5. Tránh lỗi kinh điển: **cộng** (`sum`) số lần xuất hiện của `words2` thay vì **lấy max**, chỉ xét sự có mặt của chữ cái mà bỏ qua số lần, ghi đè `need` bằng từ cuối, so sánh `>` thay vì `≥`, và dùng `set` làm mất bội số.

## 2. Đề bài & ví dụ

Cho hai mảng chuỗi `words1` và `words2`.

Chuỗi `b` là **tập con** của chuỗi `a` nếu mọi chữ cái của `b` xuất hiện trong `a` **kể cả số lần lặp**. Ví dụ `"wrr"` là tập con của `"warrior"` nhưng không phải của `"world"`.

Chuỗi `a ∈ words1` là **universal** nếu **với mọi** `b ∈ words2`, `b` là tập con của `a`.

Trả về mảng mọi chuỗi universal trong `words1`, **thứ tự bất kỳ**.

**Ví dụ 1:** `words1 = ["amazon","apple","facebook","google","leetcode"]`, `words2 = ["e","o"]` → `["facebook","google","leetcode"]`.

**Ví dụ 2:** cùng `words1`, `words2 = ["lc","eo"]` → `["leetcode"]`.

**Ví dụ 3:** `words1 = ["acaac","cccbb","aacbb","caacc","bcbbb"]`, `words2 = ["c","cc","b"]` → `["cccbb"]`.

**Ràng buộc:** `1 <= words1.length, words2.length <= 10^4`; mỗi từ dài `1..10`, chỉ gồm chữ thường; mọi từ của `words1` **khác nhau**.

In [1]:
import itertools
import random
import time
from collections import Counter
from typing import List

W1 = ["amazon", "apple", "facebook", "google", "leetcode"]
examples = [
    ((W1, ["e", "o"]), ["facebook", "google", "leetcode"]),
    ((W1, ["lc", "eo"]), ["leetcode"]),
    ((["acaac", "cccbb", "aacbb", "caacc", "bcbbb"], ["c", "cc", "b"]), ["cccbb"]),
]
for (w1, w2), want in examples:
    print(f"words2 = {w2} -> {want}")

words2 = ['e', 'o'] -> ['facebook', 'google', 'leetcode']
words2 = ['lc', 'eo'] -> ['leetcode']
words2 = ['c', 'cc', 'b'] -> ['cccbb']


## 3. Kiến thức nền

### 3.1 Tập con đa tập (multiset) và `Counter`

`Counter("warrior")` đếm số lần mỗi chữ cái: `{w:1, a:1, r:3, i:1, o:1}`. `b` là tập con của `a` ⇔ với mọi `c`: `Counter(a)[c] ≥ Counter(b)[c]`.

In [2]:
def is_subset(b, a):
    ca, cb = Counter(a), Counter(b)
    return all(ca[c] >= n for c, n in cb.items())


print(is_subset("wrr", "warrior"), is_subset("wrr", "world"), Counter("warrior"))

True False Counter({'r': 3, 'w': 1, 'a': 1, 'i': 1, 'o': 1})


### 3.2 Phép `|` và `-` trên `Counter`

- `c1 | c2`: với mỗi khoá lấy **giá trị lớn hơn** (max) — chính là phép gộp yêu cầu cần.
- `c1 - c2`: trừ theo khoá, **bỏ** các khoá có kết quả `≤ 0`. Vì vậy `not (need - Counter(a))` (kết quả rỗng) nghĩa là `a` đáp ứng mọi yêu cầu.

In [3]:
x, y = Counter("lc"), Counter("eo")
print("lc | eo =", x | y, "| (lc|eo) - leetcode =", (x | y) - Counter("leetcode"), "-> rỗng:", not ((x | y) - Counter("leetcode")))
print("(lc|eo) - amazon =", (x | y) - Counter("amazon"))

lc | eo = Counter({'l': 1, 'c': 1, 'e': 1, 'o': 1}) | (lc|eo) - leetcode = Counter() -> rỗng: True
(lc|eo) - amazon = Counter({'l': 1, 'c': 1, 'e': 1})


### 3.3 Hoán đổi "với mọi"

"Với mọi `b`, với mọi `c`: `count_a(c) ≥ count_b(c)`" ⇔ "với mọi `c`, với mọi `b`: `count_a(c) ≥ count_b(c)`" ⇔ "với mọi `c`: `count_a(c) ≥ max_b count_b(c)`". Hai vế "với mọi" cùng loại nên đổi chỗ được.

### 3.4 Chi phí

`need` chỉ có `26` phần tử: tính `need` tốn `O(Σ|b|)` (tối đa `10^5` ký tự), rồi mỗi từ `a` tốn `O(|a| + 26)`.

## 4. Ý tưởng nền tảng / trực giác

Ví dụ 2: `words2 = ["lc", "eo"]`. Từ `"leetcode"` có `l:1, e:3, t:1, c:1, o:1, d:1`. Cần `l ≥ 1`, `c ≥ 1`, `e ≥ 1`, `o ≥ 1`: đủ ✓. Từ `"facebook"`: `c:1, e:1, o:2, ...` nhưng không có `l` ✗.

Ví dụ 3: `words2 = ["c", "cc", "b"]`. Yêu cầu gộp: `c ≥ max(1, 2) = 2`, `b ≥ 1`. `"cccbb"`: `c:3 ≥ 2`, `b:2 ≥ 1` ✓. `"acaac"`: `c:2 ≥ 2`, nhưng `b:0` ✗. `"caacc"`: `c:3`, `b:0` ✗.

Quan trọng: với `["c", "cc"]` yêu cầu là `c ≥ 2` (**max**), **không** phải `c ≥ 3` (tổng): `"cc"` đã thoả cả `"c"` lẫn `"cc"`.

In [4]:
need = Counter()
for b in ["c", "cc", "b"]:
    need |= Counter(b)                                   # max theo từng chữ cái
print("yêu cầu gộp:", dict(need))
for a in ["acaac", "cccbb", "aacbb", "caacc", "bcbbb"]:
    ca = Counter(a)
    print(f"{a}: {dict(ca)} đủ? {all(ca[c] >= n for c, n in need.items())}")

yêu cầu gộp: {'c': 2, 'b': 1}
acaac: {'a': 3, 'c': 2} đủ? False
cccbb: {'c': 3, 'b': 2} đủ? True
aacbb: {'a': 2, 'c': 1, 'b': 2} đủ? False
caacc: {'c': 3, 'a': 2} đủ? False
bcbbb: {'b': 4, 'c': 1} đủ? False


## 5. Cách 1 — Brute Force: kiểm tra từng cặp

**Ý tưởng:** với mỗi `a ∈ words1`, với mỗi `b ∈ words2` kiểm tra `b` là tập con của `a` (dùng `Counter`); `a` universal nếu mọi `b` đều đạt.

In [5]:
def word_subsets_pairs(words1, words2):
    res = []
    for a in words1:
        ca = Counter(a)
        ok = True
        for b in words2:
            cb = Counter(b)
            if any(ca[c] < n for c, n in cb.items()):    # b không phải tập con của a
                ok = False
                break
        if ok:
            res.append(a)
    return res


for (w1, w2), want in examples:
    assert sorted(word_subsets_pairs(w1, w2)) == sorted(want)
print([word_subsets_pairs(w1, w2) for (w1, w2), _ in examples])

[['facebook', 'google', 'leetcode'], ['leetcode'], ['cccbb']]


In [6]:
# Code nộp LeetCode cho Cách 1 (đúng nhưng O(|words1|·|words2|·len): 10^8 kiểm tra, TLE)
class SolutionPairs:
    def wordSubsets(self, words1: List[str], words2: List[str]) -> List[str]:
        res = []
        for a in words1:
            ca = Counter(a)
            if all(all(ca[c] >= n for c, n in Counter(b).items()) for b in words2):
                res.append(a)
        return res

**Độ phức tạp:** `O(|words1| · |words2| · L)` với `L ≤ 10`: tới `10^4 · 10^4 · 10 = 10^9`. Bộ nhớ `O(1)` thêm.

**Nút thắt:** mỗi `a` lặp lại cùng một kiểm tra với **mọi** `b`. Mà điều kiện cho mọi `b` chỉ phụ thuộc vào **yêu cầu tối đa** theo từng chữ cái.

## 6. Ý tưởng thuật toán / bước đột phá

Tính một lần `need[c] = max_{b ∈ words2} count_b(c)` cho `26` chữ cái. Khi đó `a` universal ⇔ `∀c: count_a(c) ≥ need[c]`. Mỗi `a` tốn `O(|a| + 26)`; tổng `O(Σ|words2| + 26·|words1| + Σ|words1|)`.

## 7. Các cách tiếp theo

Ngoài Cách 1 (mục 5), ta có thêm 3 cách: **Cách 2** vector max bằng mảng 26, **Cách 3** phép toán `Counter`, **Cách 4** `str.count` chỉ cho chữ cái cần.

### Cách 2 — Vector yêu cầu bằng mảng 26

**Giải thích từng đoạn:**

1. `need = [0] * 26`; với mỗi `b ∈ words2`, đếm chữ cái của `b` rồi `need[c] = max(need[c], cnt_b[c])`.
2. Với mỗi `a ∈ words1`, đếm chữ cái `cnt_a` và kiểm tra `cnt_a[c] ≥ need[c]` với mọi `c`.

In [7]:
def word_subsets_array(words1, words2):
    need = [0] * 26
    for b in words2:
        cnt = [0] * 26
        for ch in b:
            cnt[ord(ch) - 97] += 1
        for c in range(26):
            if cnt[c] > need[c]:
                need[c] = cnt[c]                         # max theo từng chữ cái
    res = []
    for a in words1:
        cnt = [0] * 26
        for ch in a:
            cnt[ord(ch) - 97] += 1
        if all(cnt[c] >= need[c] for c in range(26)):
            res.append(a)
    return res


for (w1, w2), want in examples:
    assert sorted(word_subsets_array(w1, w2)) == sorted(want)
print([word_subsets_array(w1, w2) for (w1, w2), _ in examples])

[['facebook', 'google', 'leetcode'], ['leetcode'], ['cccbb']]


In [8]:
# Code nộp LeetCode cho Cách 2
class SolutionArray:
    def wordSubsets(self, words1: List[str], words2: List[str]) -> List[str]:
        need = [0] * 26
        for b in words2:
            cnt = [0] * 26
            for ch in b:
                cnt[ord(ch) - 97] += 1
            for c in range(26):
                need[c] = max(need[c], cnt[c])
        res = []
        for a in words1:
            cnt = [0] * 26
            for ch in a:
                cnt[ord(ch) - 97] += 1
            if all(cnt[c] >= need[c] for c in range(26)):
                res.append(a)
        return res

**Độ phức tạp:** `O(Σ|words2| + 26|words2| + Σ|words1| + 26|words1|)` ≈ `O(26 · (|words1| + |words2|) + tổng độ dài)`; bộ nhớ `O(26)`.

### Cách 3 — Phép toán `Counter`

**Ý tưởng:** `need = Counter(); for b in words2: need |= Counter(b)` (`|=` là gộp theo max). Từ `a` universal ⇔ `not (need - Counter(a))` (không còn yêu cầu chưa thoả).

In [9]:
def word_subsets_counter(words1, words2):
    need = Counter()
    for b in words2:
        need |= Counter(b)                               # max theo khoá
    return [a for a in words1 if not (need - Counter(a))]   # need - Counter(a) rỗng ⇔ không thiếu chữ cái nào


for (w1, w2), want in examples:
    assert sorted(word_subsets_counter(w1, w2)) == sorted(want)
print([word_subsets_counter(w1, w2) for (w1, w2), _ in examples])

[['facebook', 'google', 'leetcode'], ['leetcode'], ['cccbb']]


In [10]:
# Code nộp LeetCode cho Cách 3
class SolutionCounter:
    def wordSubsets(self, words1: List[str], words2: List[str]) -> List[str]:
        need = Counter()
        for b in words2:
            need |= Counter(b)
        return [a for a in words1 if not (need - Counter(a))]

**Độ phức tạp:** như Cách 2 (hằng số lớn hơn do `Counter`), bộ nhớ `O(26)`.

### Cách 4 — `str.count` chỉ cho chữ cái cần

**Ý tưởng:** `need` là `dict` chỉ gồm các chữ cái **thực sự cần** (`need[c] > 0`). Với mỗi `a`: `all(a.count(c) >= n for c, n in need.items())`. `str.count` chạy bằng mã C nhanh; số chữ cái cần thường nhỏ hơn 26.

In [11]:
def word_subsets_count(words1, words2):
    need = {}
    for b in words2:
        for c in set(b):
            k = b.count(c)
            if k > need.get(c, 0):
                need[c] = k                              # max theo từng chữ cái cần
    return [a for a in words1 if all(a.count(c) >= n for c, n in need.items())]


for (w1, w2), want in examples:
    assert sorted(word_subsets_count(w1, w2)) == sorted(want)
print([word_subsets_count(w1, w2) for (w1, w2), _ in examples])

[['facebook', 'google', 'leetcode'], ['leetcode'], ['cccbb']]


In [12]:
# Code nộp LeetCode cho Cách 4
class SolutionCount:
    def wordSubsets(self, words1: List[str], words2: List[str]) -> List[str]:
        need = {}
        for b in words2:
            for c in set(b):
                need[c] = max(need.get(c, 0), b.count(c))
        return [a for a in words1 if all(a.count(c) >= n for c, n in need.items())]

**Độ phức tạp:** `O(Σ|b| + |words1| · |need| · L)`; bộ nhớ `O(26)`.

## 8. TRACE từng bước

**Trace Cách 2/3** với `words2 = ["c", "cc", "b"]`:

| `b` | đếm của `b` | `need` sau |
|---|---|---|
| `c` | `{c:1}` | `{c:1}` |
| `cc` | `{c:2}` | `{c:2}` |
| `b` | `{b:1}` | `{c:2, b:1}` |

Rồi kiểm từng `a ∈ words1`:

| `a` | đếm | `c ≥ 2`? | `b ≥ 1`? | universal |
|---|---|---|---|---|
| `acaac` | `a:3, c:2` | ✓ | ✗ (0) | không |
| `cccbb` | `c:3, b:2` | ✓ | ✓ | **có** |
| `aacbb` | `a:2, c:1, b:2` | ✗ | ✓ | không |
| `caacc` | `c:3, a:2` | ✓ | ✗ | không |
| `bcbbb` | `b:4, c:1` | ✗ | ✓ | không |

Kết quả `["cccbb"]` ✓. Hàm trace:

In [13]:
def trace_need(words1, words2):
    need = Counter()
    print("gộp yêu cầu:")
    for b in words2:
        need |= Counter(b)
        print(f"  thêm {b!r}: need = {dict(sorted(need.items()))}")
    print("kiểm từng từ của words1:")
    res = []
    for a in words1:
        ca = Counter(a)
        miss = {c: n - ca[c] for c, n in need.items() if ca[c] < n}
        print(f"  {a!r}: {dict(sorted(ca.items()))} -> thiếu {miss if miss else 'không'} -> {'universal' if not miss else 'loại'}")
        if not miss:
            res.append(a)
    print("=>", res)
    return res


assert trace_need(["acaac", "cccbb", "aacbb", "caacc", "bcbbb"], ["c", "cc", "b"]) == ["cccbb"]

gộp yêu cầu:
  thêm 'c': need = {'c': 1}
  thêm 'cc': need = {'c': 2}
  thêm 'b': need = {'b': 1, 'c': 2}
kiểm từng từ của words1:
  'acaac': {'a': 3, 'c': 2} -> thiếu {'b': 1} -> loại
  'cccbb': {'b': 2, 'c': 3} -> thiếu không -> universal
  'aacbb': {'a': 2, 'b': 2, 'c': 1} -> thiếu {'c': 1} -> loại
  'caacc': {'a': 2, 'c': 3} -> thiếu {'b': 1} -> loại
  'bcbbb': {'b': 4, 'c': 1} -> thiếu {'c': 1} -> loại
=> ['cccbb']


## 9. Code nộp LeetCode

Chọn **Cách 2** (mảng 26, nhanh và rõ). Giữ nguyên `class Solution`, `wordSubsets(self, words1, words2)`:

In [14]:
class Solution:
    def wordSubsets(self, words1: List[str], words2: List[str]) -> List[str]:
        need = [0] * 26                                   # need[c] = số lần tối thiểu chữ c phải có (max theo mọi b)
        for b in words2:
            cnt = [0] * 26
            for ch in b:
                cnt[ord(ch) - 97] += 1
            for c in range(26):
                if cnt[c] > need[c]:
                    need[c] = cnt[c]
        res = []
        for a in words1:
            cnt = [0] * 26
            for ch in a:
                cnt[ord(ch) - 97] += 1
            if all(cnt[c] >= need[c] for c in range(26)):
                res.append(a)
        return res


for (w1, w2), want in examples:
    assert sorted(Solution().wordSubsets(w1, w2)) == sorted(want)
print([Solution().wordSubsets(w1, w2) for (w1, w2), _ in examples])

[['facebook', 'google', 'leetcode'], ['leetcode'], ['cccbb']]


## 10. Vì sao đúng

**(a) Định nghĩa tương đương.** `b` là tập con của `a` ⇔ `∀c: count_a(c) ≥ count_b(c)`. Từ `a` universal ⇔ `∀b ∈ words2, ∀c: count_a(c) ≥ count_b(c)`.

**(b) Đổi thứ tự lượng từ.** Hai lượng từ "với mọi" là cùng loại nên hoán đổi được: `∀c ∀b: count_a(c) ≥ count_b(c)`. Với `c` cố định, `∀b: count_a(c) ≥ count_b(c)` ⇔ `count_a(c) ≥ max_b count_b(c) = need[c]` (một số lớn hơn hoặc bằng mọi phần tử của tập hữu hạn ⇔ lớn hơn hoặc bằng cực đại). Do đó `a` universal ⇔ `∀c: count_a(c) ≥ need[c]`.

**(c) Vì sao không cộng.** Yêu cầu của các `b` **không cộng dồn**: mỗi `b` được kiểm tra **độc lập** trên cùng `a` (không "tiêu thụ" chữ cái). Với `words2 = ["c", "cc"]`, từ `"cc"` thoả cả hai dù tổng yêu cầu là `3`.

**(d) Cách 3.** `Counter.__or__` cho max theo khoá; `need - Counter(a)` giữ khoá có `need[c] − count_a(c) > 0`, tức thiếu; rỗng ⇔ không thiếu.

**(e) Cách 4.** `str.count(c)` bằng `count_a(c)`; chỉ xét các `c` có `need[c] > 0` vì `count_a(c) ≥ 0` luôn thoả khi `need[c] = 0`.

Ta kiểm chứng bằng máy: mọi cách khớp **oracle định nghĩa (từng cặp, tiêu thụ chữ cái bằng cách xoá khỏi danh sách)** trên mọi dữ liệu nhỏ (mục 12D) và ngẫu nhiên (12E); và khớp nhau ở kích thước giới hạn (12F).

## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Từng cặp | `O(\|w1\|·\|w2\|·L)` | `O(1)` | oracle một phần; TLE ở giới hạn |
| 2. Vector max (mảng 26) | `O(26(\|w1\|+\|w2\|) + ΣL)` | `O(26)` | **đáp án chính** |
| 3. Phép toán `Counter` | như Cách 2 | `O(26)` | gọn |
| 4. `str.count` | `O(ΣL + \|w1\|·\|need\|·L)` | `O(26)` | tận dụng C |

Đo thực tế (`10^4` từ mỗi mảng):

In [15]:
def rand_word(rng, alphabet, lo=1, hi=10):
    return "".join(rng.choice(alphabet) for _ in range(rng.randint(lo, hi)))


def rand_case(rng, n1, n2, alphabet="abcdefghijklmnopqrstuvwxyz", lo2=1, hi2=10):
    w1 = set()
    while len(w1) < n1:
        w1.add(rand_word(rng, alphabet))
    return sorted(w1), [rand_word(rng, alphabet, lo2, hi2) for _ in range(n2)]


def bench(fn, args, repeat=3):
    best = 1e9
    for _ in range(repeat):
        t0 = time.perf_counter()
        fn(*args)
        best = min(best, time.perf_counter() - t0)
    return best


rng = random.Random(916)
big = rand_case(rng, 10 ** 4, 10 ** 4, "abcde")
for name, fn in [("array", word_subsets_array), ("counter", word_subsets_counter), ("count", word_subsets_count)]:
    print(f"|w1| = |w2| = 10^4 {name:8s} {bench(fn, big) * 1e3:8.1f} ms")
mid = rand_case(rng, 300, 300, "abcde")
print(f"cách 1 với 300 x 300: {bench(word_subsets_pairs, mid, 1) * 1e3:8.1f} ms (tăng theo tích)")

|w1| = |w2| = 10^4 array        20.6 ms


|w1| = |w2| = 10^4 counter      39.4 ms
|w1| = |w2| = 10^4 count         8.0 ms
cách 1 với 300 x 300:      0.7 ms (tăng theo tích)


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ, (D) vét cạn mọi dữ liệu nhỏ với oracle định nghĩa, (E) ngẫu nhiên có seed + tính chất, (F) giới hạn.

### A. Ví dụ của đề

In [16]:
def oracle(words1, words2):
    """Đúng định nghĩa: b là tập con của a nếu có thể 'rút' từng chữ cái của b ra khỏi danh sách chữ cái của a; a universal nếu mọi b đều rút được."""
    def subset(b, a):
        pool = list(a)
        for ch in b:
            if ch in pool:
                pool.remove(ch)                          # mỗi chữ cái của a chỉ dùng một lần cho MỘT lần kiểm tra b
            else:
                return False
        return True

    return sorted(a for a in words1 if all(subset(b, a) for b in words2))


IMPLS = {
    "pairs": word_subsets_pairs,
    "array": word_subsets_array,
    "counter": word_subsets_counter,
    "count": word_subsets_count,
    "Solution": lambda a, b: Solution().wordSubsets(list(a), list(b)),
    "SolutionPairs": lambda a, b: SolutionPairs().wordSubsets(list(a), list(b)),
    "SolutionArray": lambda a, b: SolutionArray().wordSubsets(list(a), list(b)),
    "SolutionCounter": lambda a, b: SolutionCounter().wordSubsets(list(a), list(b)),
    "SolutionCount": lambda a, b: SolutionCount().wordSubsets(list(a), list(b)),
}


def check(w1, w2, want=None, names=None):
    if want is None:
        want = oracle(w1, w2)
    for name in (names or IMPLS):
        got = sorted(IMPLS[name](w1, w2))                # thứ tự không quan trọng: sort trước khi so
        assert got == want, (name, w1, w2, got, want)


for (w1, w2), want in examples:
    assert oracle(w1, w2) == sorted(want)
    check(w1, w2, sorted(want))
print("ví dụ đề: oracle (rút chữ cái) và", len(IMPLS), "cài đặt đúng (so sánh sau sort)")

ví dụ đề: oracle (rút chữ cái) và 9 cài đặt đúng (so sánh sau sort)


### B. Edge case

- `words2` có một từ một chữ cái.
- `words2` gồm một từ **dài hơn** mọi từ `words1` (không từ nào universal).
- Mọi từ `words1` universal (ví dụ `words2 = ["a"]` và mọi từ chứa `a`).
- Không từ nào universal.
- `words2` chứa **trùng lặp** (`["a","a","a"]` không cộng thành `aaa`).
- Yêu cầu bội số (`["aa"]`): chỉ từ có ít nhất hai `a`.
- Từ `words1` chỉ có một chữ cái.
- `words2` có yêu cầu xung đột dạng "tổng" (`["ab","bc"]`: `b` chỉ cần `1`).

In [17]:
cases = [
    (["a", "b", "ab"], ["a"], ["a", "ab"]),
    (["abc", "abcd"], ["abcdef"], []),
    (["abc", "cab", "bca"], ["a"], ["abc", "cab", "bca"]),
    (["xyz", "xxy"], ["a"], []),
    (["a", "aa", "aaa"], ["a", "a", "a"], ["a", "aa", "aaa"]),
    (["a", "aa", "aaa"], ["aa"], ["aa", "aaa"]),
    (["abc", "bcd", "abcd"], ["ab", "bc"], ["abc", "abcd"]),
    (["warrior", "world"], ["wrr"], ["warrior"]),
    (["z"], ["z"], ["z"]),
]
for w1, w2, want in cases:
    check(w1, w2, sorted(want))
print(len(cases), "edge case: đúng")

9 edge case: đúng


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [18]:
def bug_sum_counts(words1, words2):                    # cộng số lần của mọi b thay vì max
    need = Counter()
    for b in words2:
        need += Counter(b)
    return [a for a in words1 if not (need - Counter(a))]


def bug_presence_only(words1, words2):                 # chỉ xét sự có mặt của chữ cái, bỏ qua bội số
    letters = set("".join(words2))
    return [a for a in words1 if letters <= set(a)]


def bug_overwrite(words1, words2):                     # need bị ghi đè bởi từ cuối thay vì max
    need = Counter()
    for b in words2:
        need = Counter(b)
    return [a for a in words1 if not (need - Counter(a))]


def bug_strict(words1, words2):                        # dùng > thay vì >= khi so
    need = Counter()
    for b in words2:
        need |= Counter(b)
    ca_list = [(a, Counter(a)) for a in words1]
    return [a for a, ca in ca_list if all(ca[c] > n for c, n in need.items())]


def bug_only_first(words1, words2):                    # chỉ xét từ đầu tiên của words2
    need = Counter(words2[0])
    return [a for a in words1 if not (need - Counter(a))]


def bug_max_over_words1(words1, words2):               # nhầm: lấy max của words1 làm yêu cầu
    need = Counter()
    for a in words1:
        need |= Counter(a)
    return [b for b in words2 if not (Counter(b) - need)]


BUGS = [bug_sum_counts, bug_presence_only, bug_overwrite, bug_strict, bug_only_first, bug_max_over_words1]
rng = random.Random(3)
tests = []
while len(tests) < 800:
    w1 = sorted(set(rand_word(rng, "abc", 1, 4) for _ in range(rng.randint(1, 6))))
    w2 = [rand_word(rng, "abc", 1, 3) for _ in range(rng.randint(1, 3))]
    tests.append((w1, w2))
for bug in BUGS:
    bad = next(((w1, w2) for w1, w2 in tests if sorted(bug(w1, w2)) != oracle(w1, w2)), None)
    assert bad is not None, bug.__name__
    print(f"{bug.__name__:20s} sai trên words1 = {bad[0]}, words2 = {bad[1]}: ra {sorted(bug(*bad))}, đúng {oracle(*bad)}")

bug_sum_counts       sai trên words1 = ['bcaa', 'cb'], words2 = ['c', 'ca', 'aac']: ra [], đúng ['bcaa']
bug_presence_only    sai trên words1 = ['cbab'], words2 = ['aa']: ra ['cbab'], đúng []
bug_overwrite        sai trên words1 = ['abb'], words2 = ['bcb', 'cc', 'ab']: ra ['abb'], đúng []
bug_strict           sai trên words1 = ['bc', 'ccac'], words2 = ['bc']: ra [], đúng ['bc']
bug_only_first       sai trên words1 = ['bcaa', 'cb'], words2 = ['c', 'ca', 'aac']: ra ['bcaa', 'cb'], đúng ['bcaa']
bug_max_over_words1  sai trên words1 = ['bcaa', 'cb'], words2 = ['c', 'ca', 'aac']: ra ['aac', 'c', 'ca'], đúng ['bcaa']


### D. Vét cạn mọi dữ liệu nhỏ

Vũ trụ từ: mọi chuỗi độ dài `1..3` trên bảng chữ `{a, b}` (`2 + 4 + 8 = 14` từ). `words1` = **toàn bộ** vũ trụ (các từ khác nhau); `words2` = mọi danh sách độ dài `1..3` các từ trong vũ trụ (`14 + 196 + 2 744 = 2 954` danh sách, có lặp). Ngoài ra bảng chữ `{a, b, c}`, độ dài `1..2` (`3 + 9 = 12` từ): `words2` độ dài `1..3` (`12 + 144 + 1 728 = 1 884`). Oracle: rút chữ cái.

In [19]:
t0 = time.perf_counter()
cnt = 0
fast = ("pairs", "array", "counter", "count", "Solution")
for alphabet, maxlen in [("ab", 3), ("abc", 2)]:
    universe = ["".join(t) for L in range(1, maxlen + 1) for t in itertools.product(alphabet, repeat=L)]
    for k in range(1, 4):
        for w2 in itertools.product(universe, repeat=k):
            w2 = list(w2)
            want = oracle(universe, w2)
            for name in fast:
                assert sorted(IMPLS[name](universe, w2)) == want, (name, w2)
            cnt += 1
print(cnt, "bộ (words1 = toàn vũ trụ, words2): 5 cài đặt khớp oracle rút chữ cái;", round(time.perf_counter() - t0, 1), "s")
assert cnt == 2954 + 1884

4838 bộ (words1 = toàn vũ trụ, words2): 5 cài đặt khớp oracle rút chữ cái; 0.6 s


### E. Ngẫu nhiên có seed và tính chất

Dữ liệu ngẫu nhiên (bảng chữ `3..26` chữ cái), tới `60` từ. Tính chất: (1) kết quả là **tập con** của `words1` và không thay đổi khi **xáo trộn** `words2`; (2) thêm một từ vào `words2` chỉ **loại bớt** (kết quả mới ⊆ kết quả cũ); (3) lặp lại một từ của `words2` không đổi kết quả; (4) thay `words2` bằng **một từ duy nhất** gộp đủ yêu cầu `need` (từ có `need[c]` chữ `c`) cho cùng kết quả.

In [20]:
rng = random.Random(20240916)
for _ in range(1500):
    alphabet = "abcdefghijklmnopqrstuvwxyz"[:rng.randint(3, 26)]
    w1 = sorted(set(rand_word(rng, alphabet, 1, rng.randint(2, 10)) for _ in range(rng.randint(1, 60))))
    w2 = [rand_word(rng, alphabet, 1, rng.randint(1, 4)) for _ in range(rng.randint(1, 8))]
    want = oracle(w1, w2) if len(w1) * len(w2) <= 200 else sorted(word_subsets_pairs(w1, w2))
    for name in ("array", "counter", "count", "Solution", "SolutionArray", "SolutionCounter", "SolutionCount"):
        assert sorted(IMPLS[name](w1, w2)) == want, (name, w1, w2)
    assert set(want) <= set(w1)                                                           # (1)
    shuffled = w2[:]
    rng.shuffle(shuffled)
    assert sorted(Solution().wordSubsets(w1, shuffled)) == want
    extra = rand_word(rng, alphabet, 1, 4)
    assert set(Solution().wordSubsets(w1, w2 + [extra])) <= set(want)                      # (2)
    assert sorted(Solution().wordSubsets(w1, w2 + [rng.choice(w2)])) == want              # (3)
    need = Counter()
    for b in w2:
        need |= Counter(b)
    merged = "".join(c * n for c, n in need.items())
    assert sorted(Solution().wordSubsets(w1, [merged])) == want                            # (4)
print("1500 dữ liệu ngẫu nhiên: 7 cài đặt khớp; 4 tính chất (con, đơn điệu, lặp, gộp một từ) đúng")

1500 dữ liệu ngẫu nhiên: 7 cài đặt khớp; 4 tính chất (con, đơn điệu, lặp, gộp một từ) đúng


### F. Giới hạn: `10^4 × 10^4`

Các cách nhanh khớp nhau với `10^4` từ mỗi mảng; kiểm tra các kịch bản biên: yêu cầu rỗng gần như không loại; yêu cầu quá lớn loại hết.

In [21]:
rng = random.Random(500)
fast = ["array", "counter", "count", "Solution", "SolutionArray", "SolutionCounter", "SolutionCount"]
for alphabet in ("ab", "abcde", "abcdefghijklmnopqrstuvwxyz"):
    w1, w2 = rand_case(rng, 10 ** 4 if len(alphabet) > 2 else 1500, 10 ** 4, alphabet)      # bảng chữ 'ab' chỉ có 2046 từ khác nhau
    res = {k: sorted(IMPLS[k](w1, w2)) for k in fast}
    ref = res["array"]
    assert all(v == ref for v in res.values()), alphabet
    print(f"bảng chữ {len(alphabet):>2}: số từ universal = {len(ref)}")
w1, _ = rand_case(rng, 10 ** 4, 1, "abcde")
assert Solution().wordSubsets(w1, ["a"] * 10 ** 4) == [w for w in w1 if "a" in w]
assert Solution().wordSubsets(w1, ["aaaaaaaaaa", "bbbbbbbbbb"]) == [w for w in w1 if w.count("a") >= 10 and w.count("b") >= 10]
assert Solution().wordSubsets(w1, ["abcdeabcde"]) == [w for w in w1 if all(w.count(c) >= 2 for c in "abcde")]
print("biên: khớp công thức trực tiếp")

bảng chữ  2: số từ universal = 0


bảng chữ  5: số từ universal = 0


bảng chữ 26: số từ universal = 0
biên: khớp công thức trực tiếp


## 13. Các lỗi người mới hay gặp

### Lỗi 1: cộng số lần xuất hiện của `words2` thay vì lấy max

In [22]:
print(bug_sum_counts(["cc", "ccc"], ["c", "cc"]), "| đúng:", Solution().wordSubsets(["cc", "ccc"], ["c", "cc"]))
assert bug_sum_counts(["cc", "ccc"], ["c", "cc"]) == ["ccc"] and Solution().wordSubsets(["cc", "ccc"], ["c", "cc"]) == ["cc", "ccc"]

['ccc'] | đúng: ['cc', 'ccc']


**Vì sao sai:** mỗi `b` được kiểm tra **độc lập**; `"cc"` thoả cả `"c"` lẫn `"cc"`. **Đúng:** `need = max`.

### Lỗi 2: chỉ xét sự có mặt của chữ cái

In [23]:
print(bug_presence_only(["a", "aa"], ["aa"]), "| đúng:", Solution().wordSubsets(["a", "aa"], ["aa"]))
assert bug_presence_only(["a", "aa"], ["aa"]) == ["a", "aa"] and Solution().wordSubsets(["a", "aa"], ["aa"]) == ["aa"]

['a', 'aa'] | đúng: ['aa']


**Vì sao sai:** đề yêu cầu "kể cả số lần lặp"; `set` làm mất bội số. **Đúng:** đếm.

### Lỗi 3: ghi đè `need` bằng từ cuối

In [24]:
bad = next((w1, w2) for w1, w2 in tests if sorted(bug_overwrite(w1, w2)) != oracle(w1, w2))
print(bad, "->", sorted(bug_overwrite(*bad)), "| đúng:", oracle(*bad))

(['abb'], ['bcb', 'cc', 'ab']) -> ['abb'] | đúng: []


**Vì sao sai:** chỉ từ cuối được tính. **Đúng:** `need |= Counter(b)` (gộp max) hoặc `need[c] = max(need[c], ...)`.

### Lỗi 4: `>` thay vì `≥`

In [25]:
print(bug_strict(["ab", "abb"], ["ab"]), "| đúng:", Solution().wordSubsets(["ab", "abb"], ["ab"]))
assert bug_strict(["ab", "abb"], ["ab"]) == [] and Solution().wordSubsets(["ab", "abb"], ["ab"]) == ["ab", "abb"]

[] | đúng: ['ab', 'abb']


**Vì sao sai:** "có ít nhất" nghĩa là `≥`. **Đúng:** `count_a(c) >= need[c]`.

### Lỗi 5: chỉ xét từ đầu tiên của `words2`

In [26]:
bad = next((w1, w2) for w1, w2 in tests if sorted(bug_only_first(w1, w2)) != oracle(w1, w2))
print(bad, "->", sorted(bug_only_first(*bad)), "| đúng:", oracle(*bad))

(['bcaa', 'cb'], ['c', 'ca', 'aac']) -> ['bcaa', 'cb'] | đúng: ['bcaa']


**Vì sao sai:** phải thoả **mọi** `b`. **Đúng:** gộp tất cả.

### Lỗi 6 (bonus): nhầm vai trò hai mảng

Yêu cầu gộp phải lấy từ `words2` (các `b`), kiểm tra trên `words1` (các `a`), không ngược lại.

In [27]:
bad = next((w1, w2) for w1, w2 in tests if sorted(bug_max_over_words1(w1, w2)) != oracle(w1, w2))
print(bad, "->", sorted(bug_max_over_words1(*bad)), "| đúng:", oracle(*bad))

(['bcaa', 'cb'], ['c', 'ca', 'aac']) -> ['aac', 'c', 'ca'] | đúng: ['bcaa']


**Vì sao sai:** kết quả phải là các từ của `words1`. **Đúng:** `need` từ `words2`, lọc `words1`.

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "một phần tử thoả **mọi** điều kiện dạng `≥` theo từng khoá" → **gộp điều kiện bằng max** theo khoá (với `≥`) hoặc min (với `≤`), rồi kiểm một lần.

**Mẹo:**

1. Đổi thứ tự lượng từ "với mọi" để đưa max ra ngoài.
2. `Counter | Counter` = max; `Counter + Counter` = tổng; `Counter - Counter` bỏ khoá `≤ 0`.
3. Bảng chữ `26` nên dùng mảng `[0]*26` cho nhanh.
4. Phân biệt "kiểm tra độc lập từng `b`" (max) với "dùng chung tài nguyên" (tổng).
5. Oracle: rút chữ cái khỏi danh sách theo định nghĩa.

**Họ bài cùng pattern:** Find Words That Can Be Formed by Characters (1160), Ransom Note (383), Minimum Window Substring (76), Find Common Characters (1002), Permutation in String (567), Valid Anagram (242).

## 15. Biến thể

1. **Dùng chung tài nguyên:** các `b` phải **cùng** lấy chữ cái từ `a` (tổng, không phải max) — đây là `sum`.
2. **Chọn một `b` bất kỳ:** `a` thoả **ít nhất một** `b` (hoặc).
3. **Số lượng từ universal** (đếm).
4. **Yêu cầu tối thiểu** `need` (vector) trả về.
5. **Find Words That Can Be Formed by Characters (1160).**

Cài biến thể 1, 2 và đối chiếu định nghĩa:

In [28]:
def word_subsets_shared(words1, words2):
    """Biến thể 1: các b dùng CHUNG chữ cái của a (a phải chứa đồng thời mọi b): need = tổng."""
    need = Counter()
    for b in words2:
        need += Counter(b)
    return [a for a in words1 if not (need - Counter(a))]


def shared_oracle(words1, words2):
    out = []
    for a in words1:
        pool = list(a)
        ok = True
        for b in words2:
            for ch in b:
                if ch in pool:
                    pool.remove(ch)                      # tiêu thụ thật sự: lần kiểm sau không dùng lại
                else:
                    ok = False
                    break
            if not ok:
                break
        if ok:
            out.append(a)
    return sorted(out)


def word_subsets_any(words1, words2):
    """Biến thể 2: a thoả ít nhất một b."""
    needs = [Counter(b) for b in words2]
    return [a for a in words1 if any(not (nb - Counter(a)) for nb in needs)]


def any_oracle(words1, words2):
    def subset(b, a):
        pool = list(a)
        for ch in b:
            if ch in pool:
                pool.remove(ch)
            else:
                return False
        return True

    return sorted(a for a in words1 if any(subset(b, a) for b in words2))


def count_chars(words, chars):
    """Biến thể 5 (1160): tổng độ dài các từ có thể ghép từ chars (mỗi chữ dùng một lần)."""
    pool = Counter(chars)
    return sum(len(w) for w in words if not (Counter(w) - pool))


rng = random.Random(15)
for _ in range(500):
    w1, w2 = rand_case(rng, rng.randint(1, 5), rng.randint(1, 3), "abc", 1, 4)
    assert sorted(word_subsets_shared(w1, w2)) == shared_oracle(w1, w2)
    assert sorted(word_subsets_any(w1, w2)) == any_oracle(w1, w2)
assert count_chars(["cat", "bt", "hat", "tree"], "atach") == 6 and count_chars(["hello", "world", "leetcode"], "welldonehoneyr") == 10
print("biến thể dùng chung (tổng), ít nhất một (hoặc), 1160: khớp oracle trên 500 ca")

biến thể dùng chung (tổng), ít nhất một (hoặc), 1160: khớp oracle trên 500 ca


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Từng cặp | `Counter` mỗi `(a, b)` | `O(\|w1\|\|w2\|L)` | `O(1)` | oracle |
| 2. Vector max (mảng 26) | `need[c] = max_b count_b(c)` | `O(26(\|w1\|+\|w2\|) + ΣL)` | `O(26)` | **đáp án chính** |
| 3. Phép toán `Counter` | `need \|= …`; `not (need - Counter(a))` | như Cách 2 | `O(26)` | gọn |
| 4. `str.count` | chỉ chữ cái cần | `O(ΣL + \|w1\|\|need\|L)` | `O(26)` | tận dụng C |

Cả bốn khớp oracle rút-chữ-cái trên 4 838 bộ dữ liệu nhỏ và ngẫu nhiên, và khớp nhau với `10^4 × 10^4` (mục 12).

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi chạy:

In [29]:
print(Solution().wordSubsets(["abc", "aabc", "bcd"], ["ab", "ac"]), Solution().wordSubsets(["aab", "ab"], ["aa"]), Solution().wordSubsets(["x"], ["y"]))

['abc', 'aabc'] ['aab'] []


### Bài 2 (tính tay)

`words1 = ["banana", "bandana", "cabana", "nab"]`, `words2 = ["an", "na", "bn"]`. (a) Vector `need`? (b) Từ nào universal? Kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện vector max:

In [30]:
def word_subsets_todo(words1, words2):
    need = [0] * 26
    for b in words2:
        cnt = [0] * 26
        for ch in b:
            cnt[ord(ch) - 97] += 1
        pass            # TODO 1: need[c] = max(need[c], cnt[c]) cho mọi c
    res = []
    for a in words1:
        pass            # TODO 2: đếm chữ cái của a, kiểm tra cnt_a[c] >= need[c] với mọi c, nếu đạt thì res.append(a)
    return res


# Khi làm xong, bỏ comment để kiểm tra:
# assert sorted(word_subsets_todo(*examples[2][0])) == ["cccbb"]

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` bằng **phép toán `Counter`**:

In [31]:
class MySolution:
    def wordSubsets(self, words1, words2):
        pass            # TODO


# Khi làm xong:
# assert all(sorted(MySolution().wordSubsets(*a)) == sorted(w) for a, w in examples)

### Bài 5 (mini challenge): từ ngắn nhất thoả mãn

Cho `words2`, trả về một chuỗi **ngắn nhất** `m` sao cho `m` universal cho `words2` (tức mọi `b` là tập con của `m`), viết `shortest_universal(words2)`; kiểm tra rằng độ dài của nó bằng `Σ need[c]` và mọi `b` là tập con của nó.

In [32]:
def shortest_universal(words2):
    pass            # TODO


# Khi làm xong:
# m = shortest_universal(["lc", "eo"]); assert len(m) == 4 and all(not (Counter(b) - Counter(m)) for b in ["lc", "eo"])

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: `need` = max theo chữ cái của `words2`.
- Mức 2: `["ab","ac"]` → `a:1, b:1, c:1`; `["aa"]` → `a:2`.
- Mức 3: kết quả `['abc', 'aabc'] ['aab'] []` (`bcd` không có chữ `a`; `ab` chỉ có một `a` nên không đạt `aa`).

**Bài 2**

- Mức 1: `an` → `a:1,n:1`; `na` → cùng; `bn` → `b:1,n:1`.
- Mức 2: `need = {a:1, n:1, b:1}`.
- Mức 3: `banana` (`b,a,n`), `bandana`, `cabana` và `nab` đều chứa đủ `a`, `n`, `b` nên cả bốn từ đều universal.

**Bài 3**

- Mức 1: `need[c] = max(need[c], cnt[c])`.
- Mức 2: đếm `a` thành `cnt` mảng 26.
- Mức 3: `all(cnt[c] >= need[c] for c in range(26))`.

**Bài 4**

- Mức 1: `need = Counter(); need |= Counter(b)`.
- Mức 2: `not (need - Counter(a))`.
- Mức 3: danh sách thu gọn.

**Bài 5**

- Mức 1: xây `need` (max).
- Mức 2: từ ngắn nhất cần đúng `need[c]` chữ `c`.
- Mức 3: `"".join(c * n for c, n in need.items())`.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert`.

### Bài 1

In [33]:
assert Solution().wordSubsets(["abc", "aabc", "bcd"], ["ab", "ac"]) == ["abc", "aabc"]
assert sorted(Solution().wordSubsets(["abc", "aabc", "bcd"], ["ab", "ac"])) == oracle(["abc", "aabc", "bcd"], ["ab", "ac"])
assert Solution().wordSubsets(["aab", "ab"], ["aa"]) == ["aab"] and Solution().wordSubsets(["x"], ["y"]) == []
print("Bài 1 OK:", Solution().wordSubsets(["abc", "aabc", "bcd"], ["ab", "ac"]), ["aab"], [])

Bài 1 OK: ['abc', 'aabc'] ['aab'] []


### Bài 2

(a) `need = {a:1, n:1, b:1}`. (b) `banana`, `bandana`, `cabana`, `nab` đều có `a`, `n`, `b` ⇒ cả bốn từ universal.

In [34]:
w1, w2 = ["banana", "bandana", "cabana", "nab"], ["an", "na", "bn"]
need = Counter()
for b in w2:
    need |= Counter(b)
assert dict(need) == {"a": 1, "n": 1, "b": 1}
assert sorted(Solution().wordSubsets(w1, w2)) == sorted(w1) == oracle(w1, w2)
print("Bài 2 OK")

Bài 2 OK


### Bài 3

In [35]:
def word_subsets_answer(words1, words2):
    need = [0] * 26
    for b in words2:
        cnt = [0] * 26
        for ch in b:
            cnt[ord(ch) - 97] += 1
        for c in range(26):
            need[c] = max(need[c], cnt[c])
    res = []
    for a in words1:
        cnt = [0] * 26
        for ch in a:
            cnt[ord(ch) - 97] += 1
        if all(cnt[c] >= need[c] for c in range(26)):
            res.append(a)
    return res


assert sorted(word_subsets_answer(*examples[2][0])) == ["cccbb"]
for w1, w2 in tests[:300]:
    assert sorted(word_subsets_answer(w1, w2)) == oracle(w1, w2)
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [36]:
class MySolutionAnswer:
    def wordSubsets(self, words1, words2):
        need = Counter()
        for b in words2:
            need |= Counter(b)
        return [a for a in words1 if not (need - Counter(a))]


assert all(sorted(MySolutionAnswer().wordSubsets(*a)) == sorted(w) for a, w in examples)
for w1, w2 in tests[:300]:
    assert sorted(MySolutionAnswer().wordSubsets(w1, w2)) == oracle(w1, w2)
print("Bài 4 OK")

Bài 4 OK


### Bài 5

In [37]:
def shortest_universal_answer(words2):
    need = Counter()
    for b in words2:
        need |= Counter(b)
    return "".join(c * n for c, n in need.items())


m = shortest_universal_answer(["lc", "eo"])
assert len(m) == 4 and all(not (Counter(b) - Counter(m)) for b in ["lc", "eo"])
rng = random.Random(5)
for _ in range(300):
    w2 = [rand_word(rng, "abc", 1, 4) for _ in range(rng.randint(1, 4))]
    m = shortest_universal_answer(w2)
    assert all(not (Counter(b) - Counter(m)) for b in w2)
    # tối thiểu: bỏ đi một chữ cái bất kỳ thì có b không còn là tập con
    for i in range(len(m)):
        shorter = m[:i] + m[i + 1:]
        assert any(Counter(b) - Counter(shorter) for b in w2)
print("Bài 5 OK: từ gộp đủ yêu cầu và tối thiểu")

Bài 5 OK: từ gộp đủ yêu cầu và tối thiểu


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (thoả mọi điều kiện `≥` theo khoá → gộp bằng max, kiểm một lần):**

```python
class Solution:
    def wordSubsets(self, words1: List[str], words2: List[str]) -> List[str]:
        need = Counter()
        for b in words2:
            need |= Counter(b)                      # max theo từng chữ cái
        return [a for a in words1 if not (need - Counter(a))]
```

**Checklist trước khi submit:**

- [ ] Gộp `words2` bằng **max** (không cộng)?
- [ ] Kiểm tra `count_a(c) >= need[c]` (có `=`)?
- [ ] Xét đủ **mọi** từ của `words2` (không chỉ từ đầu)?
- [ ] Xét số lần lặp (không chỉ sự có mặt)?
- [ ] Lọc `words1` (không phải `words2`); thứ tự bất kỳ?
- [ ] Đã thử 3 ví dụ, `words2` có từ trùng lặp, yêu cầu bội số (`["aa"]`), không từ nào universal?
- [ ] Đúng tên `class Solution`, `wordSubsets(self, words1, words2)`, không in debug.